## Feature Engineering

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC

# Load the raw dataset
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset shape: {df.shape}')

Dataset shape: (7043, 21)


In [31]:
# Convert TotalCharges from string to numeric, coercing blank strings to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many NaN values were created
print(f'Missing TotalCharges: {df["TotalCharges"].isna().sum()}')

# Fill NaN with the median value
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f'Missing after imputation: {df["TotalCharges"].isna().sum()}')

Missing TotalCharges: 11
Missing after imputation: 0


In [32]:
# Drop customerID since it has no predictive value
df = df.drop(columns=['customerID'])

# Derived feature: average monthly charge (guards against division by zero)
df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)

# Derived feature: binary flag for premium support services
df['HasPremiumSupport'] = (
    (df['OnlineSecurity'] == 'Yes') | (df['TechSupport'] == 'Yes')
).astype(int)

print('New columns: AvgMonthlyCharge, HasPremiumSupport')
print(df[['AvgMonthlyCharge', 'HasPremiumSupport']].head())

New columns: AvgMonthlyCharge, HasPremiumSupport
   AvgMonthlyCharge  HasPremiumSupport
0         14.925000                  0
1         53.985714                  1
2         36.050000                  1
3         40.016304                  1
4         50.550000                  0


In [33]:
# Encode the target column as 0/1
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# One-hot encode all remaining object columns
cat_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f'Shape after encoding: {df.shape}')
print(f'Churn distribution:')
print(df['Churn'].value_counts())

Shape after encoding: (7043, 33)
Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [34]:
import sys
sys.path.insert(0, '..')
from src.preprocessing import preprocess_data

# Reload the raw data and run the full pipeline
df_raw = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
X_train_res, X_test, y_train_res, y_test, feature_names = preprocess_data(df_raw)

print(f'Training set (resampled): {X_train_res.shape}')
print(f'Test set: {X_test.shape}')
print('Training label distribution:')
print(y_train_res.value_counts())

Training set (resampled): (8278, 32)
Test set: (1409, 32)
Training label distribution:
Churn
0    4139
1    4139
Name: count, dtype: int64


In [35]:
# Save processed datasets for the modeling notebook
X_train_res.to_csv('../data/processed/X_train_resampled.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train_res.to_csv('../data/processed/y_train_resampled.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Saved to data/processed/')

Saved to data/processed/
